# FrameWeave Ovis2.5-9B — baseline `frame_only` (train)

Colab training notebook (**A100 40 GB**). Biến thể `frame_only` giữ question encoder, token projection, question-conditioned pooling trong từng frame và residual head. Toàn bộ global relation và local correspondence được bỏ: không có $r_i^G$, $r_{ip}^L$, $\beta$, $\pi$, $\gamma$. Notebook chỉ gọi `run_ablation.py ... --variant frame_only` trong `/content/ovis_env`.

$$
k_{ip} = W_k\,\mathrm{LN}_x(x_{ip}), \qquad
\alpha_{ip} = \operatorname{softmax}_{p:\,m_{ip}=1}\!\left(\frac{k_{ip}^\top q}{\sqrt r}\right)
$$

$$
s_i = f_i\,\mathrm{LN}_s\!\Big(W_v \sum_p \alpha_{ip}\,x_{ip}\Big), \qquad
\bar s_i = s_i + f_i\,W_{\mathrm{abs}}[\tau_i;\rho_i]
$$

$$
h^F_{ip} = \mathrm{SiLU}\big(B_T k_{ip} + B_F \bar s_i + B_q q + b_h\big), \qquad
u^F_{ip} = B_o h^F_{ip} + b_o, \qquad
a^F_{ip} = \sigma\big(w_g^\top h^F_{ip} + b_g\big)
$$

$$
\widetilde x^F_{ip} = x_{ip} + m_{ip}\,\chi_i\,a^F_{ip}\,\kappa^F_{ip}\,u^F_{ip}
$$

- Softmax và pooling chỉ chạy trên token của frame $i$. Mỗi frame giữ thời gian tuyệt đối $(\tau_i,\rho_i)$ của chính nó, ngang lượng thông tin thời gian của Token MLP. Relative-time feature giữa các frame được bỏ.
- $\kappa^F_{ip}$ là detached RMS calibration của bài, tính với proposal $u^F_{ip}$. $B_o, b_o$ khởi tạo 0, gate khởi tạo 0.5, $W_{\mathrm{abs}}$ khởi tạo 0. Objective V6 giữ nguyên.
- $\chi_i = 1$ khi frame $i$ hợp lệ và còn ít nhất một frame hợp lệ khác. Bỏ relation giữ nguyên $\chi_i$; một frame hợp lệ duy nhất vẫn là identity.
- 4,405,505 tham số trainable (Hybrid 5,589,253). Width giữ $r=256$.

## Protocol giống suite sáu biến thể

Backbone [`ATH-MaaS/Ovis2.5-9B`](https://huggingface.co/ATH-MaaS/Ovis2.5-9B) đóng băng, revision `5a1f9b995c98ae5b85a28ec89d88d74295be0ea7`. Adapter fp32, thinking off, 448×448. Split V6: 454 câu / 128 video train, 67 câu / 19 video val. Loss $\mathrm{logsumexp}(\mathcal L_{\mathrm{cand}},\mathcal L_{\mathrm{corr}})-\log 2$, AdamW lr $2\cdot10^{-4}$, weight decay 0.01, cosine với warmup 10%, accumulation 8, $K\in\{3,\dots,8\}$. Chọn epoch có validation $L$ thấp nhất, patience 3, tối đa 8 epoch. Cell 6 kiểm tra `schedule_sha256` bằng của suite (`5423f680…`): cùng câu hỏi, thứ tự, $K$, frame sampling và hoán vị đáp án.

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `ovis/reljoin`, `ablation/`, `configs/` và `run_ablation.py` vào `/content`. Không upload file source.

Notebook tải PerceptionTest, `prepare` ghi split vào output. Model vào `/content/checkpoints/ovis2_5_9b`, video vào `/content/perceptiontest_lite`. Checkpoint: `/content/outputs/frameweave_ovis_ablation/seed_42/frame_only/best/adapter.pt`. Drive riêng: `/content/drive/MyDrive/frameweave_ovis_frame_only_outputs`, để archive `frameweave_ovis_ablation_outputs` của sáu biến thể giữ nguyên.

## `RUN_MODE`

Chạy lần lượt `unit`, `smoke`, `train`.

`unit` chạy trên CPU, không tải model hay video. `smoke` là hai update thật để kiểm tra gradient; số smoke nằm ngoài bảng paper. `train` resume được từ Drive riêng với `RESUME=True`.

Eval: `FrameWeave_Ovis_FrameOnly_HERBench_Eval.ipynb`.


## 1. Configuration

**Input:** none · **Output:** `RUN_MODE`, pins, paths


In [ ]:
# 1. Configuration — change RUN_MODE / RESUME here
from __future__ import annotations
from pathlib import Path

RUN_MODE = "unit"  # unit | smoke | train
VARIANT = "frame_only"
RESUME = False
SEED = 42
REQUIRE_SUITE_SCHEDULE = True

ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."
assert VARIANT == "frame_only", "Notebook này chỉ train frame_only. Suite sáu biến thể dùng FrameWeave_Ovis_Ablation_Experiment.ipynb."
assert SEED == 42, "Hybrid của suite là seed 42."

MODEL_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
TRAIN_DATASET_ID = "lmms-lab/LLaVA-Video-178K"
TRAIN_DATASET_REVISION = "6d8c562dc26d70042a0d9704d1cae58c94b89098"
SUITE_SCHEDULE_SHA256 = "5423f6802d7bb772ec7819927c7a7827ee3cdca856ba62b5107a885ab5fd605d"
FRAME_ONLY_PARAMS = 4_405_505

VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
ABLATION = ROOT
OUT_DIR = ABLATION / "outputs" / "frameweave_ovis_ablation"
RUN_DIR = OUT_DIR / f"seed_{SEED}" / VARIANT
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_frame_only_outputs")

print({
    "RUN_MODE": RUN_MODE,
    "VARIANT": VARIANT,
    "RESUME": RESUME,
    "SEED": SEED,
    "RUN_DIR": str(RUN_DIR),
    "PERSIST": str(PERSIST),
    "suite_schedule": SUITE_SCHEDULE_SHA256[:12],
    "backbone": MODEL_ID,
})


## 2. Environment installation (`ovis_env`)

Official Ovis2.5-9B pins: `torch==2.4.0`, `transformers==4.51.3`, `flash-attn==2.7.0.post2`.
Stop if flash-attn install fails — the ViT hard-imports it.


In [ ]:
# 2. Bootstrap isolated ovis_env (Python 3.11 + torch 2.4.0 + flash-attn 2.7)
from __future__ import annotations
import os, subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "ovis_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs" / "frameweave_ovis_ablation" / "logs" / "cell4_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc

sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "datasets", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging",
])
print("Installing flash-attn==2.7.0.post2 (required by Ovis ViT)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, flash_attn; "
    "print(torch.__version__); print(transformers.__version__); print(flash_attn.__version__); "
    "print(torch.__file__); print(transformers.__file__)"
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/ovis_env/" in line, f"Package leaked outside venv: {line}"
print("ovis_env ready:", PYTHON)
print("LOG:", LOG)


In [ ]:
# 2b. Extra pip (same packages as the V6 notebook). Writes a log so a failure is inspectable.
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
LOG_DIR = ROOT / "outputs" / "frameweave_ovis_ablation" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / "cell5_pip.log"

if not PYTHON.exists():
    msg = (
        f"Missing {PYTHON}. Run the ovis_env bootstrap cell first "
        "(section 2), then re-run this cell.\n"
    )
    LOG.write_text(msg, encoding="utf-8")
    print(msg)
    raise FileNotFoundError(msg)

cmd = [
    str(PYTHON), "-m", "pip", "install",
    "--disable-pip-version-check",
    "--upgrade-strategy", "only-if-needed",
    "scikit-learn==1.3.2",
    "matplotlib==3.8.4",
    "numpy==1.25.0",
]
print("$", " ".join(cmd))
proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_text = f"$ {' '.join(cmd)}\nexit={proc.returncode}\n\n{proc.stdout or ''}"
LOG.write_text(log_text, encoding="utf-8")
print(proc.stdout or "")
print("LOG:", LOG)

persist = Path("/content/drive/MyDrive/frameweave_ovis_frame_only_outputs")
if persist.parent.exists():
    try:
        persist.mkdir(parents=True, exist_ok=True)
        dest = persist / "logs" / LOG.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(log_text, encoding="utf-8")
        print("Copied log →", dest)
    except OSError as exc:
        print("Persist copy skipped:", exc)

if proc.returncode != 0:
    raise RuntimeError(f"cell 5 pip failed ({proc.returncode}). See {LOG}")
print("cell 5 pip OK")


## 3. Reproducibility — paths, Drive riêng

Output local vẫn là `/content/outputs/frameweave_ovis_ablation/`. Drive là `frameweave_ovis_frame_only_outputs`. `RESUME=True` copy Drive riêng về output trước khi train.


In [ ]:
# 3. Paths, Drive. Resume copies the frame_only tree back before a new train.
from __future__ import annotations
import os
import shutil
from pathlib import Path

ROOT = Path("/content")
ABLATION = ROOT
OUT_DIR = ABLATION / "outputs" / "frameweave_ovis_ablation"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_frame_only_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    PERSIST = OUT_DIR
PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)
os.environ["OVIS_ABLATION_ROOT"] = str(ROOT)

try:
    RESUME
except NameError:
    RESUME = False
if RESUME and PERSIST.resolve() != OUT_DIR.resolve() and any(PERSIST.iterdir()):
    shutil.copytree(PERSIST, OUT_DIR, dirs_exist_ok=True)
    print("Restored", PERSIST, "→", OUT_DIR)

print({
    "RELJOIN_PERSIST": str(PERSIST),
    "OUT_DIR": str(OUT_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "RESUME": RESUME,
})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


link(REPO_DIR / "ovis" / "reljoin", ROOT / "reljoin")
link(REPO_DIR / "ovis_ablation" / "ablation", ROOT / "ablation")
link(REPO_DIR / "ovis_ablation" / "configs", ROOT / "configs")
link(REPO_DIR / "ovis_ablation" / "run_ablation.py", ROOT / "run_ablation.py")
print("FrameWeave source linked from", REPO_DIR / "ovis_ablation", "and", REPO_DIR / "ovis" / "reljoin")


In [ ]:
# 4. Verify FrameWeave sources. ablation/ must be the version with frame_only.
from __future__ import annotations
from pathlib import Path

ROOT = Path("/content")
required = [
    ROOT / "run_ablation.py",
    ROOT / "configs" / "ovis_ablations.yaml",
    ROOT / "ablation" / "modules.py",
    ROOT / "ablation" / "config.py",
    ROOT / "ablation" / "train.py",
    ROOT / "ablation" / "invariants.py",
    ROOT / "ablation" / "eval_herbench.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "train.py",
    ROOT / "reljoin" / "losses.py",
    ROOT / "reljoin" / "data_perception.py",
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
markers = {
    ROOT / "ablation" / "modules.py": "class FrameOnlyAdapter",
    ROOT / "ablation" / "config.py": 'EXTRA_VARIANTS = ("frame_only",)',
    ROOT / "ablation" / "train.py": "frame_pooler.token_key.weight",
    ROOT / "ablation" / "invariants.py": "def test_frame_only_frame_isolation",
}
stale = [str(path) for path, marker in markers.items() if marker not in path.read_text(encoding="utf-8")]
assert not stale, "ablation/ thiếu frame_only. Restart runtime rồi Run All để pull repo:\n" + "\n".join(stale)
print("sources OK", len(required), "| frame_only markers OK", len(markers))


## 5. Dataset / model download (gated by `RUN_MODE`)

`unit` tải không gì. `smoke` và `train` tải Ovis vào `/content/checkpoints/ovis2_5_9b` và PerceptionTest vào `/content/perceptiontest_lite`. Cell sau đó `prepare` mới ghi split 454/67. Notebook train không tải HERBench.


In [ ]:
# 5. Conditional downloads (no HERBench in this training notebook)
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell 2 trước"
os.environ["PATH"] = f"{ROOT / 'ovis_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"

MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
MODEL_HF_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
need_data = RUN_MODE in {"smoke", "train"}
print({"need_model": need_data, "need_perception": need_data, "RUN_MODE": RUN_MODE})

if need_data:
    print("Downloading Ovis2.5-9B (~18 GB bf16)...")
    MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    py = (
        "import os\n"
        "os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'\n"
        "from huggingface_hub import snapshot_download\n"
        f"snapshot_download(repo_id='{MODEL_HF_ID}', revision='{MODEL_REVISION}', "
        f"local_dir=r'{MODEL_DIR}', ignore_patterns=['*.msgpack','*.h5','flax_model*','tf_model*'])\n"
        "print('model OK')\n"
    )
    subprocess.run([str(PYTHON), "-c", py], check=True)
    assert (MODEL_DIR / "config.json").exists(), f"Missing {MODEL_DIR / 'config.json'}"

    print("Downloading PerceptionTest train shard...")
    perception = (
        "import sys\n"
        "from pathlib import Path\n"
        f"sys.path.insert(0, {str(ROOT)!r})\n"
        "from reljoin.data_perception import download_perception_lite\n"
        "ann, videos = download_perception_lite(Path('/content/perceptiontest_lite'))\n"
        "print(ann)\n"
        "print(videos)\n"
    )
    subprocess.run([str(PYTHON), "-c", perception], check=True)

print("Download step done for RUN_MODE=", RUN_MODE)


## 6. Prepare shared schedule, then unit tests

`prepare` dựng lại split và schedule bằng đúng code của suite, rồi cell so `schedule_sha256` với suite. `unit` chạy trên CPU và gồm hai kiểm tra của `frame_only`:

- $\chi_i$ theo số frame hợp lệ: residual khác 0 đúng ở token hợp lệ của frame có frame hợp lệ khác; một frame hợp lệ và frame bị mask giữ nguyên.
- Cách ly frame: sau khi $W_o \ne 0$, giữ question, timestamps, masks và đổi nội dung frame $j \ne i$; output adapter của frame $i$ bằng bitwise, output frame $j$ đổi. Kiểm tra ở projected-token interface.


In [ ]:
# 6. prepare + unit. Split 454/128 and 67/19, schedule identical to the suite.
from __future__ import annotations
import json
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
ABLATION = ROOT
OUT = ABLATION / "outputs" / "frameweave_ovis_ablation"
assert PYTHON.exists(), "Chạy cell 2 trước"
try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"
try:
    REQUIRE_SUITE_SCHEDULE
except NameError:
    REQUIRE_SUITE_SCHEDULE = True
SUITE_SCHEDULE_SHA256 = "5423f6802d7bb772ec7819927c7a7827ee3cdca856ba62b5107a885ab5fd605d"
FRAME_ONLY_PARAMS = 4_405_505

for stage in ("prepare", "unit"):
    print("$", PYTHON, "run_ablation.py", stage, flush=True)
    subprocess.run([str(PYTHON), "run_ablation.py", stage], cwd=str(ABLATION), check=True)

meta_path = OUT / "shared" / "schedule_meta.json"
if RUN_MODE in {"smoke", "train"}:
    assert meta_path.is_file(), "prepare did not write the V6 split after PerceptionTest download"
if meta_path.is_file():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print({
        "train_questions": meta["train_questions"],
        "train_videos": meta["train_videos"],
        "val_questions": meta["val_questions"],
        "val_videos": meta["val_videos"],
        "schedule_sha256": meta["schedule_sha256"][:12],
        "frames_ready": meta["frames"]["rows_ready"],
        "frames_total": meta["frames"]["rows_total"],
    })
    assert meta["train_questions"] == 454 and meta["train_videos"] == 128
    assert meta["val_questions"] == 67 and meta["val_videos"] == 19
    same_schedule = meta["schedule_sha256"] == SUITE_SCHEDULE_SHA256
    print("schedule giống suite:", same_schedule)
    if REQUIRE_SUITE_SCHEDULE:
        assert same_schedule, (
            f"schedule_sha256 {meta['schedule_sha256'][:12]} khác suite {SUITE_SCHEDULE_SHA256[:12]}. "
            "K, hoán vị đáp án hoặc frame plan đã đổi, nên frame_only sẽ không train cùng dữ liệu với Hybrid. "
            "Kiểm tra bản PerceptionTest và ovis_env."
        )
else:
    print("unit: split 454/67 is written when smoke/train has downloaded PerceptionTest")

report = json.loads((OUT / "unit" / "invariant_report.json").read_text(encoding="utf-8"))
assert report["ok"] is True
budget = json.loads((OUT / "shared" / "parameter_budget.json").read_text(encoding="utf-8"))
frame_only = budget["frame_only"]
print("frame_only params", {
    "trainable": frame_only["trainable"],
    "relative_to_hybrid": round(frame_only["relative_to_hybrid"], 4),
})
assert frame_only["trainable"] == FRAME_ONLY_PARAMS, frame_only["trainable"]
print("Unit invariants passed, including chi and frame isolation for frame_only.")


## 7. Smoke or train

Một process, một backbone, chỉ `frame_only`. `smoke` chạy hai update thật và yêu cầu gradient tới `frame_pooler.token_key.weight` khác 0 sau update thứ hai. `train` ghi `seed_42/frame_only/best/adapter.pt` theo validation $L$ thấp nhất.


In [ ]:
# 7. Smoke or train frame_only (gated by RUN_MODE)
from __future__ import annotations
import json
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
ABLATION = ROOT
OUT = ABLATION / "outputs" / "frameweave_ovis_ablation"
VARIANT = "frame_only"

try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"
try:
    RESUME
except NameError:
    RESUME = False
try:
    SEED
except NameError:
    SEED = 42

if RUN_MODE == "unit":
    print("RUN_MODE=unit — skip smoke/train")
else:
    assert RUN_MODE in {"smoke", "train"}, RUN_MODE
    cmd = [str(PYTHON), "run_ablation.py", RUN_MODE, "--variant", VARIANT]
    if RUN_MODE == "train" and RESUME:
        cmd.append("--resume")
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(ABLATION), check=True)
    if RUN_MODE == "smoke":
        report = json.loads((OUT / "smoke" / "smoke_report.json").read_text(encoding="utf-8"))
        assert report["status"] == "ok", report
        print(report["variants"])
        print("Smoke finished. paper_result is false; số smoke nằm ngoài bảng paper.")
    else:
        run_dir = OUT / f"seed_{SEED}" / VARIANT
        best = run_dir / "best" / "adapter.pt"
        assert best.is_file(), f"Train kết thúc nhưng thiếu {best}. Xem log ở trên."
        selected = json.loads((run_dir / "selected.json").read_text(encoding="utf-8"))
        print({"selected_epoch": selected.get("epoch"), "val_loss": selected.get("val_loss"), "checkpoint": str(best)})


## 8. Validation $L$ and train-loss curves


In [ ]:
# 8. Plot frame_only val.jsonl and train.jsonl. Empty until train has written them.
from __future__ import annotations
import json
from pathlib import Path

ROOT = Path("/content")
try:
    SEED
except NameError:
    SEED = 42
OUT = ROOT / "outputs" / "frameweave_ovis_ablation"
run_dir = OUT / f"seed_{SEED}" / "frame_only"


def read_rows(path: Path) -> list[dict]:
    if not path.is_file():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


val_rows = read_rows(run_dir / "val.jsonl")
updates = [row for row in read_rows(run_dir / "train.jsonl") if row.get("event") == "update"]
if not val_rows and not updates:
    print("No frame_only outputs yet. Run train first.")
else:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot([row["epoch"] for row in val_rows], [row["val_loss"] for row in val_rows], marker="o", label="frame_only")
    axes[1].plot([row["global_step"] for row in updates], [row["loss"] for row in updates], label="frame_only")
    selected_path = run_dir / "selected.json"
    if selected_path.is_file():
        selected = json.loads(selected_path.read_text(encoding="utf-8"))
        axes[0].axvline(selected.get("epoch"), color="gray", linestyle="--", label=f"selected epoch {selected.get('epoch')}")
        print({"selected_epoch": selected.get("epoch"), "val_loss": selected.get("val_loss")})
    axes[0].set_title("frame_only validation L")
    axes[0].set_xlabel("epoch")
    axes[1].set_title("frame_only train loss per update")
    axes[1].set_xlabel("global step")
    axes[0].legend(fontsize=8)
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    dest = run_dir / "loss_curves.png"
    fig.savefig(dest, dpi=120)
    print("Wrote", dest)
    plt.show()


## 9. Export artifacts

Copy output sang Drive `frameweave_ovis_frame_only_outputs` và zip. Notebook eval đọc `seed_42/frame_only/` từ Drive này.


In [ ]:
# 9. Copy outputs to the frame_only Drive + zip
from __future__ import annotations
import shutil
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path("/content")
OUT = ROOT / "outputs" / "frameweave_ovis_ablation"
persist = Path("/content/drive/MyDrive/frameweave_ovis_frame_only_outputs")
if not persist.parent.exists():
    persist = OUT
persist.mkdir(parents=True, exist_ok=True)
if OUT.exists() and persist.resolve() != OUT.resolve():
    shutil.copytree(OUT, persist, dirs_exist_ok=True)
    print("Copied", OUT, "→", persist)

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = ROOT / f"frameweave_ovis_frame_only_{stamp}"
if OUT.exists():
    shutil.make_archive(str(zip_path), "zip", root_dir=OUT)
    print("Zip:", f"{zip_path}.zip")
print("Persist:", persist)
